# 02 – Limpeza e preparação dos dados (PySpark)

Aqui tratamos os problemas encontrados no notebook `01_compreensao_e_qualidade`. Cada decisão vem justificada logo acima do código.

O pré-processamento é feito com **Apache Spark** (PySpark + Spark SQL): a mesma lógica roda numa máquina só ou num cluster, então o fluxo continua funcionando se a base crescer. A exploração do notebook 01 ficou em pandas, porque lá só olhamos os dados.

No final, conferimos que os problemas foram resolvidos (antes × depois) e salvamos o dataset do RA1 em `data/processed/gatos_ra1.csv`, com **uma linha por passagem do gato pelo abrigo** (entrada + saída; as passagens sem saída registrada ficam marcadas como `sem_saida`).

> Lembrete de escopo: aqui só preparamos os dados. Análises, agregações e conclusões ficam para o RA2/RA3.

In [1]:
import os
import sys

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# garante que o Spark use o mesmo Python do notebook (no Windows ele pode pegar outro)
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("adocao-gatinhos-ra1")
    # fuso fixo: sem isso o Spark usa o fuso do computador e horários de horário de verão podiam virar nulo
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

## 1. Carregar os dados brutos e filtrar os gatos

Lemos os CSVs sem `inferSchema`, então tudo vem como texto, igual ao notebook 01. Já na leitura:
- trocamos os nomes das colunas para snake_case, porque nomes com espaço dão trabalho no Spark SQL;
- criamos uma coluna `linha` com a ordem original do arquivo, que usamos depois para escolher entre registros repetidos de forma reproduzível.

In [2]:
def ler_csv(caminho):
    df = spark.read.csv(caminho, header=True)
    novos_nomes = [col.lower().replace(" ", "_") for col in df.columns]
    return df.toDF(*novos_nomes).withColumn("linha", F.monotonically_increasing_id())


entradas_raw = ler_csv("../data/raw/intakes.csv")
saidas_raw = ler_csv("../data/raw/outcomes.csv")

entradas = entradas_raw.filter(F.col("animal_type") == "Cat")
saidas = saidas_raw.filter(F.col("animal_type") == "Cat")

# guarda o bruto dos gatos pra comparar no final
entradas_antes = entradas
saidas_antes = saidas

print("Entradas de gatos:", entradas.count())
print("Saídas de gatos:  ", saidas.count())
entradas.printSchema()

Entradas de gatos: 69324


Saídas de gatos:   69399
root
 |-- animal_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- datetime: string (nullable = true)
 |-- monthyear: string (nullable = true)
 |-- found_location: string (nullable = true)
 |-- intake_type: string (nullable = true)
 |-- intake_condition: string (nullable = true)
 |-- animal_type: string (nullable = true)
 |-- sex_upon_intake: string (nullable = true)
 |-- age_upon_intake: string (nullable = true)
 |-- breed: string (nullable = true)
 |-- color: string (nullable = true)
 |-- linha: long (nullable = false)



## 2. Duplicados

**Decisão:** remover as linhas idênticas, porque são o mesmo registro repetido e contariam o gato duas vezes. A coluna `linha` fica de fora da comparação, já que ela é diferente em cada linha.

Nas entradas, também removemos os registros com o mesmo `animal_id` e o mesmo `datetime`. Um gato não entra duas vezes no mesmo minuto, e as duas versões só mudam o endereço ou o tipo de entrada. Ficamos com a que aparece **primeiro no arquivo**: não há como saber qual está certa, e são só 4 casos.

> No Spark o `dropDuplicates` com algumas colunas fica com uma linha qualquer. Por isso usamos uma *window* com `row_number`, ordenada pela `linha` original.

In [3]:
colunas_ent = [c for c in entradas.columns if c != "linha"]
colunas_sai = [c for c in saidas.columns if c != "linha"]

entradas = entradas.dropDuplicates(colunas_ent)
saidas = saidas.dropDuplicates(colunas_sai)

primeira = Window.partitionBy("animal_id", "datetime").orderBy("linha")
entradas = (
    entradas.withColumn("rn", F.row_number().over(primeira))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Entradas:", entradas_antes.count(), "->", entradas.count())
print("Saídas:  ", saidas_antes.count(), "->", saidas.count())

Entradas: 69324 -> 69309


Saídas:   69399 -> 69387


## 3. Conversão das datas

**Decisão:** converter `datetime` para timestamp, informando o formato de cada base (eles são diferentes). Nas saídas, o fuso `-05:00` só aparece em linhas à meia-noite, sem horário real, então basta pegar os 19 primeiros caracteres.

In [4]:
entradas = entradas.withColumn(
    "data_hora_entrada", F.to_timestamp("datetime", "MM/dd/yyyy hh:mm:ss a")
)
saidas = saidas.withColumn(
    "data_hora_saida", F.to_timestamp(F.substring("datetime", 1, 19), "yyyy-MM-dd'T'HH:mm:ss")
)

# se alguma data não tivesse convertido, ia aparecer como nulo aqui
print("Datas inválidas nas entradas:", entradas.filter(F.col("data_hora_entrada").isNull()).count())
print("Datas inválidas nas saídas:  ", saidas.filter(F.col("data_hora_saida").isNull()).count())

Datas inválidas nas entradas: 0


Datas inválidas nas saídas:   0


## 4. Idade em meses

**Decisão:** transformar o texto em número de **meses**. Meses é uma boa escala porque a maioria dos gatos é filhote (idades em semanas), mas também há gatos de 20 anos.

- Usamos 1 mês = 30,44 dias e 1 ano = 365,25 dias (médias do calendário).
- "1 week" e "1 weeks" viram a mesma coisa (tiramos o "s" do final).
- **Idade negativa** é impossível, então vira nulo.
- **"0 years"** vira 0. No notebook 01 vimos que a data de nascimento desses gatos é igual à data de entrada e que quase todos são `Nursing`/`Neonatal`, ou seja, recém-nascidos.

In [5]:
numero = F.split("age_upon_intake", " ").getItem(0).cast("int")
unidade = F.regexp_replace(F.split("age_upon_intake", " ").getItem(1), "s$", "")

dias_de_vida = (
    F.when(unidade == "day", numero)
    .when(unidade == "week", numero * 7)
    .when(unidade == "month", numero * 30.44)
    .when(unidade == "year", numero * 365.25)
)

entradas = entradas.withColumn(
    "idade_entrada_meses",
    F.when(dias_de_vida >= 0, F.round(dias_de_vida / 30.44, 1)),  # negativa fica nula
)

entradas.select("age_upon_intake", "idade_entrada_meses").distinct().orderBy("idade_entrada_meses").show(8)

+---------------+-------------------+
|age_upon_intake|idade_entrada_meses|
+---------------+-------------------+
|       -4 years|               NULL|
|          1 day|                0.0|
|        0 years|                0.0|
|         4 days|                0.1|
|         3 days|                0.1|
|         2 days|                0.1|
|         6 days|                0.2|
|         1 week|                0.2|
+---------------+-------------------+
only showing top 8 rows


In [6]:
entradas.select(
    F.sum(F.col("idade_entrada_meses").isNull().cast("int")).alias("idades_nulas"),
    F.min("idade_entrada_meses").alias("menor"),
    F.max("idade_entrada_meses").alias("maior"),
).show()

+------------+-----+-----+
|idades_nulas|menor|maior|
+------------+-----+-----+
|           1|  0.0|264.0|
+------------+-----+-----+



## 5. Pareamento de cada entrada com a sua saída

Esse é o passo mais importante. O mesmo gato pode ter várias entradas e saídas, então não dá pra juntar só pelo `animal_id`.

**Regra:** a saída de uma entrada é a **primeira saída do mesmo gato depois dela**, desde que aconteça **antes da próxima entrada** desse gato. Se não existe saída nesse intervalo, a saída dessa passagem não foi registrada.

Antes de parear, numeramos as passagens de cada gato (1ª entrada, 2ª entrada...) e guardamos a data da próxima entrada com `lead`.

In [7]:
por_gato = Window.partitionBy("animal_id").orderBy("data_hora_entrada")

entradas = (
    entradas
    .withColumn("passagem", F.row_number().over(por_gato))
    .withColumn("proxima_entrada", F.lead("data_hora_entrada").over(por_gato))
)

entradas.filter(F.col("animal_id") == "A774102") \
    .select("animal_id", "passagem", "data_hora_entrada", "proxima_entrada").show()

+---------+--------+-------------------+-------------------+
|animal_id|passagem|  data_hora_entrada|    proxima_entrada|
+---------+--------+-------------------+-------------------+
|  A774102|       1|2018-06-11 12:53:00|2018-09-07 16:55:00|
|  A774102|       2|2018-09-07 16:55:00|2018-09-28 11:36:00|
|  A774102|       3|2018-09-28 11:36:00|2018-11-28 15:01:00|
|  A774102|       4|2018-11-28 15:01:00|2019-01-07 14:02:00|
|  A774102|       5|2019-01-07 14:02:00|2019-01-25 17:24:00|
|  A774102|       6|2019-01-25 17:24:00|2019-02-21 12:13:00|
|  A774102|       7|2019-02-21 12:13:00|               NULL|
+---------+--------+-------------------+-------------------+



**1ª rodada (Spark SQL):** juntamos cada entrada com as saídas do mesmo gato que ficam entre ela e a próxima entrada, e ficamos com a primeira (`ROW_NUMBER`).

In [8]:
entradas.createOrReplaceTempView("entradas")
saidas.createOrReplaceTempView("saidas")

rodada1 = spark.sql("""
    SELECT *
    FROM (
        SELECT e.animal_id, e.passagem, s.data_hora_saida, s.outcome_type, s.outcome_subtype,
               ROW_NUMBER() OVER (PARTITION BY e.animal_id, e.passagem ORDER BY s.data_hora_saida) AS rn
        FROM entradas e
        JOIN saidas s
          ON  s.animal_id = e.animal_id
          AND s.data_hora_saida >= e.data_hora_entrada
          AND (e.proxima_entrada IS NULL OR s.data_hora_saida < e.proxima_entrada)
    )
    WHERE rn = 1
""").drop("rn")

rodada1.createOrReplaceTempView("rodada1")
print("Pareadas na 1ª rodada:", rodada1.count())

Pareadas na 1ª rodada: 68566


**2ª rodada:** no notebook 01 vimos que algumas saídas têm o **horário** registrado antes da entrada, mas sempre no **mesmo dia**. Como o horário das saídas não é confiável e a data é, para as entradas que ficaram sem par aceitamos uma saída **do mesmo dia** que ainda não tenha sido usada. Cada entrada e cada saída só podem ser usadas uma vez.

In [9]:
rodada2 = spark.sql("""
    WITH entradas_sem_par AS (
        SELECT e.* FROM entradas e
        LEFT ANTI JOIN rodada1 r ON r.animal_id = e.animal_id AND r.passagem = e.passagem
    ),
    saidas_livres AS (
        SELECT s.* FROM saidas s
        LEFT ANTI JOIN rodada1 r ON r.animal_id = s.animal_id AND r.data_hora_saida = s.data_hora_saida
    ),
    candidatos AS (
        SELECT e.animal_id, e.passagem, e.data_hora_entrada,
               s.data_hora_saida, s.outcome_type, s.outcome_subtype,
               ROW_NUMBER() OVER (PARTITION BY e.animal_id, e.passagem ORDER BY s.data_hora_saida) AS rn_entrada
        FROM entradas_sem_par e
        JOIN saidas_livres s
          ON s.animal_id = e.animal_id
         AND to_date(s.data_hora_saida) = to_date(e.data_hora_entrada)
    )
    SELECT animal_id, passagem, data_hora_saida, outcome_type, outcome_subtype
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY animal_id, data_hora_saida ORDER BY data_hora_entrada) AS rn_saida
        FROM candidatos
        WHERE rn_entrada = 1
    )
    WHERE rn_saida = 1
""")

print("Pareadas na 2ª rodada (mesmo dia):", rodada2.count())

Pareadas na 2ª rodada (mesmo dia): 333


In [10]:
saidas_pareadas = rodada1.unionByName(rodada2)
pares = entradas.join(saidas_pareadas, on=["animal_id", "passagem"], how="left")

print("Entradas:", pares.count())
print("Total pareado:", pares.filter(F.col("data_hora_saida").isNotNull()).count())

# confere: nenhuma saída foi usada por duas entradas
saidas_pareadas.groupBy("animal_id", "data_hora_saida").count().filter("count > 1").count()

Entradas: 69309


Total pareado: 68899


0

**Entradas que ficaram sem saída**

Quantas são e em que ano entraram? Também calculamos a porcentagem das entradas de cada ano que ficou sem saída.

In [11]:
por_ano = (
    pares.groupBy(F.year("data_hora_entrada").alias("ano"))
    .agg(
        F.count("*").alias("entradas"),
        F.sum(F.col("data_hora_saida").isNull().cast("int")).alias("sem_saida"),
    )
    .withColumn("pct_sem_saida", F.round(100 * F.col("sem_saida") / F.col("entradas"), 1))
    .orderBy("ano")
)
por_ano.show()

+----+--------+---------+-------------+
| ano|entradas|sem_saida|pct_sem_saida|
+----+--------+---------+-------------+
|2013|    1309|        0|          0.0|
|2014|    6937|        8|          0.1|
|2015|    7484|        3|          0.0|
|2016|    6632|        3|          0.0|
|2017|    6488|        4|          0.1|
|2018|    6018|        0|          0.0|
|2019|    7880|        2|          0.0|
|2020|    3448|        0|          0.0|
|2021|    5112|        0|          0.0|
|2022|    4684|        0|          0.0|
|2023|    5600|        4|          0.1|
|2024|    6046|       14|          0.2|
|2025|    1671|      372|         22.3|
+----+--------+---------+-------------+



A maior parte é de 2025: são gatos que **ainda estavam no abrigo** quando a base parou de ser atualizada (05/05/2025). Nos outros anos são poucos casos, provavelmente saídas que não foram registradas.

**Decisão:** **manter** essas passagens no dataset, com `desfecho = "sem_saida"` e `data_saida`/`dias_no_abrigo` vazios.

Remover parece natural, porque sem saída não há tempo nem desfecho. Mas isso criaria um **viés**. Os gatos que ainda estavam no abrigo em maio/2025 são justamente os que estavam demorando, e eles são uma fatia grande das entradas de 2025 (ver `pct_sem_saida`). Sem eles, os anos mais recentes pareceriam ter estadias mais curtas do que de fato tiveram. Esse tipo de dado é chamado de **censurado**: sabemos que o gato ficou *pelo menos* até 05/05/2025, só não sabemos quando saiu.

Mantendo e marcando essas passagens, o RA2 decide como tratá-las: excluir de forma consciente, limitar o período analisado ou usar técnicas próprias para dados censurados.

In [12]:
# cache: sem ele o Spark refaz o pareamento inteiro a cada count() daqui pra frente
pares = pares.cache()
print("Passagens:", pares.count())
print("  com saída:", pares.filter(F.col("data_hora_saida").isNotNull()).count())
print("  sem saída:", pares.filter(F.col("data_hora_saida").isNull()).count())

Passagens: 69309


  com saída: 68899


  sem saída: 410


Já as saídas que não tiveram entrada correspondente (gatos que entraram antes de out/2013, quando a base começa, por exemplo) ficam de fora. Sem a entrada não temos as características do gato na chegada nem a data de início.

## 6. Datas finais e dias no abrigo

**Decisão:** guardar só a **data** (sem horário) de entrada e de saída e calcular `dias_no_abrigo` como a diferença em dias (`datediff`). O horário das saídas não é confiável (seção 3.5 do notebook 01), então usar horas criaria tempos negativos ou imprecisos. Um gato que entra e sai no mesmo dia fica com 0 dias. Nas passagens sem saída, os dias ficam vazios.

**Atenção para o RA2:** como vimos no notebook 01 (seção 3.13), nas adoções com subtipo `Foster` esse tempo provavelmente inclui dias em lar temporário. Por isso o `subtipo_desfecho` fica no dataset.

In [13]:
pares = (
    pares
    .withColumn("data_entrada", F.to_date("data_hora_entrada"))
    .withColumn("data_saida", F.to_date("data_hora_saida"))
    .withColumn("dias_no_abrigo", F.datediff("data_saida", "data_entrada"))
)

print("Tempos negativos:", pares.filter(F.col("dias_no_abrigo") < 0).count())
pares.select("dias_no_abrigo").summary("count", "mean", "min", "25%", "50%", "75%", "max").show()

Tempos negativos: 0


+-------+------------------+
|summary|    dias_no_abrigo|
+-------+------------------+
|  count|             68899|
|   mean|23.205242456349147|
|    min|                 0|
|    25%|                 2|
|    50%|                 7|
|    75%|                31|
|    max|              1745|
+-------+------------------+



## 7. Outliers de tempo no abrigo

**Decisão:** **manter** todas as passagens e criar a flag `estadia_acima_1_ano`.

- A regra do IQR não serve aqui: a distribuição é muito assimétrica e ela marcaria como outlier gatos que ficaram 2 ou 3 meses, o que é normal num abrigo.
- As estadias acima de 1 ano são poucas (menos de 0,2%). Não temos como saber se são reais ou se faltou registrar uma saída e uma volta no meio. Remover seria arriscado: se forem reais, são justamente os gatos que mais demoraram pra ser adotados.
- Com a flag, no RA2 dá pra rodar as análises com e sem esses casos, ou usar medidas robustas como a mediana.

In [14]:
pares = pares.withColumn("estadia_acima_1_ano", F.col("dias_no_abrigo") > 365)

pares.groupBy("estadia_acima_1_ano").count().show()
pares.filter("estadia_acima_1_ano") \
    .select("animal_id", "data_entrada", "data_saida", "dias_no_abrigo", "outcome_type") \
    .orderBy(F.desc("dias_no_abrigo")).show(5)

+-------------------+-----+
|estadia_acima_1_ano|count|
+-------------------+-----+
|               true|  123|
|              false|68776|
|               NULL|  410|
+-------------------+-----+



+---------+------------+----------+--------------+------------+
|animal_id|data_entrada|data_saida|dias_no_abrigo|outcome_type|
+---------+------------+----------+--------------+------------+
|  A764237|  2017-12-26|2022-10-06|          1745|    Adoption|
|  A702369|  2015-05-12|2019-05-21|          1470|    Adoption|
|  A707965|  2015-07-20|2019-05-28|          1408|    Adoption|
|  A762152|  2017-11-15|2021-04-03|          1235|    Adoption|
|  A669236|  2013-12-17|2016-12-23|          1102|    Adoption|
+---------+------------+----------+--------------+------------+
only showing top 5 rows


**Idade:** a idade máxima é de cerca de 22 anos. É alta, mas possível para um gato, então **mantemos**.

## 8. Nome

**Decisão:** criar duas colunas, sem apagar informação:
- `tem_nome`: o gato chegou com um nome de verdade. É `False` quando o nome está ausente **ou** quando no lugar do nome está o peso do filhote ("103 Grams"), o próprio ID ("A840400") ou só um número ("#2"). Nenhum desses é um nome.
- `nome_com_asterisco`: o nome começa com `*`. Costuma-se dizer que isso indica nome dado pelo abrigo, mas o portal não documenta isso. Guardamos a informação separada para decidir no RA2 como usar.

In [15]:
# (?i) = ignora maiúsculas/minúsculas
placeholder = r"(?i)^\*?(?:A\d{5,}|#?\d+|\d+\s*(?:g|grams)\b.*|grams\s*\d+)$"
eh_placeholder = F.coalesce(F.col("name").rlike(placeholder), F.lit(False))

pares = (
    pares
    .withColumn("tem_nome", F.col("name").isNotNull() & ~eh_placeholder)
    .withColumn("nome_com_asterisco", F.coalesce(F.col("name").startswith("*"), F.lit(False)))
)

print("Nomes que eram peso/ID/número:", pares.filter(eh_placeholder).count())
pares.groupBy("tem_nome").pivot("nome_com_asterisco").count().show()

Nomes que eram peso/ID/número: 2895


+--------+-----+-----+
|tem_nome|false| true|
+--------+-----+-----+
|    true|18474|20845|
|   false|29858|  132|
+--------+-----+-----+



## 9. Sexo e castração

**Decisão:** separar `sex_upon_intake` em duas colunas:
- `sexo`: macho / femea / desconhecido
- `castrado`: true (Neutered/Spayed), false (Intact) ou nulo quando é `Unknown`

Usamos a coluna da **entrada** e não a da saída: o abrigo castra muitos gatos antes da adoção, então a castração na saída já seria consequência da passagem pelo abrigo. A pergunta 4 é sobre o gato que **chega** castrado ou não. Como vimos no notebook 01, o sexo é igual nas duas bases.

In [16]:
primeira_palavra = F.split("sex_upon_intake", " ").getItem(0)
ultima_palavra = F.element_at(F.split("sex_upon_intake", " "), -1)

pares = (
    pares
    .withColumn("sexo", F.when(ultima_palavra == "Male", "macho")
                         .when(ultima_palavra == "Female", "femea")
                         .otherwise("desconhecido"))
    .withColumn("castrado", F.when(primeira_palavra.isin("Neutered", "Spayed"), True)
                             .when(primeira_palavra == "Intact", False))
)

pares.groupBy("sex_upon_intake", "sexo", "castrado").count().orderBy("sex_upon_intake").show()

+---------------+------------+--------+-----+
|sex_upon_intake|        sexo|castrado|count|
+---------------+------------+--------+-----+
|  Intact Female|       femea|   false|26302|
|    Intact Male|       macho|   false|25082|
|  Neutered Male|       macho|    true| 6720|
|  Spayed Female|       femea|    true| 6174|
|        Unknown|desconhecido|    NULL| 5031|
+---------------+------------+--------+-----+



## 10. Faixa etária

**Decisão:** dividir a idade em três faixas, como na pergunta 2:
- **filhote**: menos de 12 meses
- **adulto**: de 1 a menos de 10 anos
- **idoso**: 10 anos ou mais

Os cortes seguem as fases de vida do gato das diretrizes da AAHA/AAFP (*Feline Life Stage Guidelines*, 2021), que consideram filhote até 1 ano e sênior acima de 10 anos. Juntamos as fases intermediárias ("jovem adulto" e "adulto maduro") em uma só.

In [17]:
idade = F.col("idade_entrada_meses")
pares = pares.withColumn(
    "faixa_etaria",
    F.when(idade < 12, "filhote").when(idade < 120, "adulto").when(idade >= 120, "idoso"),
)
pares.groupBy("faixa_etaria").count().orderBy(F.desc("count")).show()

+------------+-----+
|faixa_etaria|count|
+------------+-----+
|     filhote|45783|
|      adulto|21521|
|       idoso| 2004|
|        NULL|    1|
+------------+-----+



## 11. Raça

**Decisão:**
1. Ficar só com a **primeira raça** quando há duas separadas por `/`.
2. Tirar o sufixo " Mix". Na prática "Domestic Shorthair" e "Domestic Shorthair Mix" são o mesmo tipo de gato sem raça definida, e o "Mix" só multiplica as categorias.
3. Agrupar em **"Outra"** as raças com menos de 100 passagens (~0,15% da base). Com tão poucos gatos não dá pra comparar nada no RA2.

Para contar quantas passagens cada categoria tem sem sair do DataFrame, usamos `count` numa *window* particionada pela própria categoria.

In [18]:
def agrupa_raras(df, coluna, minimo=100):
    contagem = F.count("*").over(Window.partitionBy(coluna))
    return df.withColumn(coluna, F.when(contagem >= minimo, F.col(coluna)).otherwise("Outra"))


pares = pares.withColumn(
    "raca", F.trim(F.regexp_replace(F.split("breed", "/").getItem(0), " Mix", ""))
)
pares = agrupa_raras(pares, "raca")

print("Raças antes:", pares.select("breed").distinct().count(), "-> depois:", pares.select("raca").distinct().count())
pares.groupBy("raca").count().orderBy(F.desc("count")).show()

Raças antes: 111 -> depois: 10


+--------------------+-----+
|                raca|count|
+--------------------+-----+
|  Domestic Shorthair|57856|
|Domestic Medium Hair| 5457|
|   Domestic Longhair| 2488|
|             Siamese| 2046|
|               Outra|  445|
|  American Shorthair|  286|
|            Snowshoe|  285|
|          Maine Coon|  162|
|                Manx|  144|
|        Russian Blue|  140|
+--------------------+-----+



## 12. Cor principal e `is_black`

**Decisão:**
- `cor_principal`: a primeira cor antes da `/` ("Brown Tabby/White" → "Brown Tabby"). Cores com menos de 100 passagens viram "Outra", pelo mesmo motivo das raças. Os nomes das cores ficam em inglês, como no portal: muitos são padrões de pelagem sem tradução direta (Tabby, Tortie, Torbie, Calico...).
- `is_black`: `true` só quando o gato é **todo preto**: cor "Black" ou "Black/Black" (a mesma cor repetida, que aparece em cerca de 100 registros). A crença popular da pergunta 1 é sobre o gato preto. "Black/White" (preto e branco) e "Black Tabby" (rajado) têm outra aparência e ficam como `false`, mas continuam aparecendo com cor principal "Black". Assim, no RA2 dá pra usar as duas definições.

In [19]:
pares = pares.withColumn("cor_principal", F.split("color", "/").getItem(0))
pares = agrupa_raras(pares, "cor_principal")
pares = pares.withColumn("is_black", F.col("color").isin("Black", "Black/Black"))

print("Cores antes:", pares.select("color").distinct().count(), "-> depois:", pares.select("cor_principal").distinct().count())
pares.groupBy("cor_principal").count().orderBy(F.desc("count")).show(30)

Cores antes: 327 -> depois: 26


+---------------+-----+
|  cor_principal|count|
+---------------+-----+
|    Brown Tabby|16356|
|          Black|15386|
|   Orange Tabby| 7726|
|           Blue| 4368|
|     Blue Tabby| 4122|
|          White| 4041|
|         Tortie| 3406|
|         Calico| 3268|
|         Torbie| 2628|
|    Cream Tabby| 1890|
|     Lynx Point| 1037|
|     Gray Tabby|  755|
|     Seal Point|  698|
|          Outra|  470|
|    Flame Point|  458|
|         Orange|  456|
|           Gray|  417|
|    Black Tabby|  384|
|    Black Smoke|  285|
|    Lilac Point|  211|
|   Tortie Point|  202|
|   Silver Tabby|  188|
|          Cream|  185|
|     Blue Point|  136|
|          Brown|  124|
|Chocolate Point|  112|
+---------------+-----+



In [20]:
# conferindo: só os gatos todo pretos viram is_black
pares.filter(F.col("cor_principal") == "Black") \
    .groupBy("color", "is_black").count().orderBy(F.desc("count")).show(6)

+------------+--------+-----+
|       color|is_black|count|
+------------+--------+-----+
|       Black|    true| 8974|
| Black/White|   false| 6232|
| Black/Black|    true|  109|
|  Black/Gray|   false|   17|
| Black/Brown|   false|   13|
|Black/Orange|   false|   10|
+------------+--------+-----+
only showing top 6 rows


## 13. Tipo de entrada, condição de entrada e desfecho

**Tipo de entrada:** a coluna já está limpa. Só traduzimos para português.

**Condição de entrada:** são 18 categorias, muitas raríssimas.
- As principais são traduzidas. `Nursing` fica como está, porque o portal não diz se é o filhote que ainda mama ou a mãe que amamenta.
- `Medical`, `Med Attn` e `Med Urgent` viram **atendimento_medico**. Os três indicam necessidade de atenção médica e só aparecem nos anos mais recentes, o que sugere variações de nome no cadastro.
- As outras categorias (menos de 1% cada) viram **outra**.
- Observação para o RA2: `Nursing` some em 2022 e `Neonatal` começa em 2021, então comparações entre anos nessas categorias precisam de cuidado.

**Desfecho:** traduzimos as categorias claras. `Rto-Adopt` e `Disposal` não são explicadas no portal, então ficam com o nome original pra não inventarmos um significado. As poucas saídas sem desfecho viram **desconhecido**, e as passagens sem saída (seção 5) viram **sem_saida**. O `outcome_subtype` fica como veio, em `subtipo_desfecho`, porque é um detalhe complementar.

Para traduzir usamos um dicionário Python transformado num mapa do Spark (`create_map`). O que não está no dicionário recebe um valor padrão.

In [21]:
def traduz(coluna, dicionario, padrao):
    mapa = F.create_map(*[F.lit(x) for par in dicionario.items() for x in par])
    return F.coalesce(mapa[F.col(coluna)], F.lit(padrao))


pares = (
    pares
    .withColumn("tipo_entrada", traduz("intake_type", {
        "Stray": "perdido",
        "Owner Surrender": "entregue_pelo_dono",
        "Public Assist": "assistencia_publica",
        "Abandoned": "abandonado",
        "Euthanasia Request": "pedido_de_eutanasia",
    }, None))
    .withColumn("condicao_entrada", traduz("intake_condition", {
        "Normal": "normal",
        "Injured": "ferido",
        "Sick": "doente",
        "Nursing": "nursing",
        "Neonatal": "neonatal",
        "Medical": "atendimento_medico",
        "Med Attn": "atendimento_medico",
        "Med Urgent": "atendimento_medico",
    }, "outra"))
    .withColumn("desfecho", F.when(F.col("data_hora_saida").isNull(), "sem_saida").otherwise(traduz("outcome_type", {
        "Adoption": "adocao",
        "Transfer": "transferencia",
        "Return to Owner": "devolvido_ao_dono",
        "Euthanasia": "eutanasia",
        "Died": "morte",
        "Missing": "desaparecido",
        "Relocate": "realocado",
        "Rto-Adopt": "rto-adopt",
        "Disposal": "disposal",
    }, "desconhecido")))
    .withColumn("subtipo_desfecho", F.col("outcome_subtype"))
)

In [22]:
for coluna in ["tipo_entrada", "condicao_entrada", "desfecho"]:
    pares.groupBy(coluna).count().orderBy(F.desc("count")).show()

+-------------------+-----+
|       tipo_entrada|count|
+-------------------+-----+
|            perdido|53610|
| entregue_pelo_dono|13442|
|assistencia_publica| 1449|
|         abandonado|  748|
|pedido_de_eutanasia|   60|
+-------------------+-----+



+------------------+-----+
|  condicao_entrada|count|
+------------------+-----+
|            normal|57019|
|            ferido| 4339|
|            doente| 3285|
|           nursing| 2559|
|          neonatal| 1467|
|             outra|  389|
|atendimento_medico|  251|
+------------------+-----+



+-----------------+-----+
|         desfecho|count|
+-----------------+-----+
|           adocao|35433|
|    transferencia|26577|
|devolvido_ao_dono| 2857|
|        eutanasia| 2555|
|            morte|  998|
|        sem_saida|  410|
|        rto-adopt|  249|
|         disposal|  164|
|     desaparecido|   51|
|     desconhecido|    9|
|        realocado|    6|
+-----------------+-----+



## 14. Montando o dataset final

Ficamos só com as colunas limpas, com nomes em português e snake_case. As colunas que não entram:
- `found_location`: não usada nas perguntas.
- `monthyear`: redundante.
- `date_of_birth`: tem datas impossíveis e a idade já vem da entrada.
- o próprio nome do gato: para as perguntas basta saber se ele tem nome.

In [23]:
colunas = [
    "animal_id", "passagem", "tem_nome", "nome_com_asterisco",
    "data_entrada", "data_saida", "dias_no_abrigo", "estadia_acima_1_ano",
    "idade_entrada_meses", "faixa_etaria", "sexo", "castrado",
    "raca", "cor_principal", "is_black",
    "tipo_entrada", "condicao_entrada", "desfecho", "subtipo_desfecho",
]

gatos = pares.select(colunas).orderBy("data_entrada", "animal_id", "passagem").cache()
gatos.printSchema()
gatos.show(5)

root
 |-- animal_id: string (nullable = true)
 |-- passagem: integer (nullable = false)
 |-- tem_nome: boolean (nullable = false)
 |-- nome_com_asterisco: boolean (nullable = false)
 |-- data_entrada: date (nullable = true)
 |-- data_saida: date (nullable = true)
 |-- dias_no_abrigo: integer (nullable = true)
 |-- estadia_acima_1_ano: boolean (nullable = true)
 |-- idade_entrada_meses: double (nullable = true)
 |-- faixa_etaria: string (nullable = true)
 |-- sexo: string (nullable = false)
 |-- castrado: boolean (nullable = true)
 |-- raca: string (nullable = true)
 |-- cor_principal: string (nullable = true)
 |-- is_black: boolean (nullable = true)
 |-- tipo_entrada: string (nullable = true)
 |-- condicao_entrada: string (nullable = false)
 |-- desfecho: string (nullable = false)
 |-- subtipo_desfecho: string (nullable = true)



+---------+--------+--------+------------------+------------+----------+--------------+-------------------+-------------------+------------+------------+--------+------------------+-------------+--------+------------------+----------------+-------------+----------------+
|animal_id|passagem|tem_nome|nome_com_asterisco|data_entrada|data_saida|dias_no_abrigo|estadia_acima_1_ano|idade_entrada_meses|faixa_etaria|        sexo|castrado|              raca|cor_principal|is_black|      tipo_entrada|condicao_entrada|     desfecho|subtipo_desfecho|
+---------+--------+--------+------------------+------------+----------+--------------+-------------------+-------------------+------------+------------+--------+------------------+-------------+--------+------------------+----------------+-------------+----------------+
|  A664235|       1|   false|             false|  2013-10-01|2013-10-01|             0|              false|                0.2|     filhote|desconhecido|    NULL|Domestic Shorthair|   

## 15. Checagem final: antes × depois

In [24]:
entradas_antes.createOrReplaceTempView("entradas_antes")
saidas_antes.createOrReplaceTempView("saidas_antes")
gatos.createOrReplaceTempView("gatos")

# pares simples do bruto (gatos com 1 entrada e 1 saída), só pra medir "saída antes da entrada" no antes
saida_antes_entrada = spark.sql("""
    WITH e AS (
        SELECT animal_id, to_timestamp(datetime, 'MM/dd/yyyy hh:mm:ss a') AS t
        FROM entradas_antes WHERE animal_id IN (SELECT animal_id FROM entradas_antes GROUP BY animal_id HAVING count(*) = 1)
    ),
    s AS (
        SELECT animal_id, to_timestamp(substring(datetime, 1, 19), "yyyy-MM-dd'T'HH:mm:ss") AS t
        FROM saidas_antes WHERE animal_id IN (SELECT animal_id FROM saidas_antes GROUP BY animal_id HAVING count(*) = 1)
    )
    SELECT count(*) FROM e JOIN s USING (animal_id) WHERE s.t < e.t
""").first()[0]


def conta(df, condicao):
    return df.filter(condicao).count()


checagem = [
    ("Duplicados exatos (entradas + saídas)",
     entradas_antes.count() - entradas_antes.dropDuplicates(colunas_ent).count()
     + saidas_antes.count() - saidas_antes.dropDuplicates(colunas_sai).count(),
     gatos.count() - gatos.dropDuplicates().count()),
    ("Mesma entrada registrada 2x (ID + data/hora)",
     entradas_antes.dropDuplicates(colunas_ent).count()
     - entradas_antes.dropDuplicates(["animal_id", "datetime"]).count(),
     entradas.count() - entradas.dropDuplicates(["animal_id", "datetime"]).count()),
    ("Saída antes da entrada", saida_antes_entrada, conta(gatos, "dias_no_abrigo < 0")),
    ("Idade negativa", conta(entradas_antes, "age_upon_intake LIKE '-%'"), conta(gatos, "idade_entrada_meses < 0")),
    ("Colunas de data guardadas como texto",
     sum(tipo == "string" for nome, tipo in entradas_antes.dtypes + saidas_antes.dtypes if nome == "datetime"),
     sum(tipo == "string" for nome, tipo in gatos.dtypes if nome in ("data_entrada", "data_saida"))),
    ("Categorias de condição de entrada",
     entradas_antes.select("intake_condition").distinct().count(), gatos.select("condicao_entrada").distinct().count()),
    ("Categorias de cor", entradas_antes.select("color").distinct().count(), gatos.select("cor_principal").distinct().count()),
    ("Categorias de raça", entradas_antes.select("breed").distinct().count(), gatos.select("raca").distinct().count()),
    ("Desfecho ausente", conta(saidas_antes, "outcome_type IS NULL"), conta(gatos, "desfecho = 'desconhecido'")),
    ("Categorias de sexo (misturadas com castração)",
     entradas_antes.select("sex_upon_intake").distinct().count(), gatos.select("sexo").distinct().count()),
]
spark.createDataFrame(checagem, ["problema", "antes", "depois"]).show(truncate=False)

+---------------------------------------------+-----+------+
|problema                                     |antes|depois|
+---------------------------------------------+-----+------+
|Duplicados exatos (entradas + saídas)        |23   |0     |
|Mesma entrada registrada 2x (ID + data/hora) |4    |0     |
|Saída antes da entrada                       |327  |0     |
|Idade negativa                               |1    |0     |
|Colunas de data guardadas como texto         |2    |0     |
|Categorias de condição de entrada            |18   |7     |
|Categorias de cor                            |327  |26    |
|Categorias de raça                           |111  |10    |
|Desfecho ausente                             |9    |9     |
|Categorias de sexo (misturadas com castração)|5    |3     |
+---------------------------------------------+-----+------+



Notas sobre a tabela:
- "Mesma entrada registrada 2x": o "depois" é medido nas entradas já limpas, antes do pareamento.
- "Colunas de data guardadas como texto": as datas vinham como texto em três formatos diferentes (entradas, saídas sem fuso e saídas com fuso). Agora são do tipo `date`.
- "Categorias de sexo": antes eram 5 valores misturando sexo e castração; agora são 3 (macho, femea, desconhecido) e a castração está em `castrado`.
- "Desfecho ausente" não vai a zero de propósito: os poucos casos ficaram marcados como `desconhecido`, em vez de sumirem.

In [25]:
# nulos por coluna no dataset final
gatos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas]).toPandas().T \
    .rename(columns={0: "nulos"}).query("nulos > 0")

,nulos
data_saida,410
dias_no_abrigo,410
estadia_acima_1_ano,410
idade_entrada_meses,1
faixa_etaria,1
castrado,5031
subtipo_desfecho,28995


Os únicos ausentes que sobram são esperados:
- `data_saida`, `dias_no_abrigo` e `estadia_acima_1_ano`: as passagens com `desfecho = "sem_saida"` (seção 5).
- `idade_entrada_meses` e `faixa_etaria`: o gato com idade negativa.
- `castrado`: gatos com sexo `Unknown`, em que a castração não é conhecida.
- `subtipo_desfecho`: detalhe opcional que o abrigo nem sempre preenche.

In [26]:
gatos.select(
    F.min("data_entrada").alias("primeira_entrada"),
    F.max("data_saida").alias("ultima_saida"),
    F.count("*").alias("passagens"),
    F.countDistinct("animal_id").alias("gatos_diferentes"),
).show()
gatos.select("dias_no_abrigo", "idade_entrada_meses").summary().show()

+----------------+------------+---------+----------------+
|primeira_entrada|ultima_saida|passagens|gatos_diferentes|
+----------------+------------+---------+----------------+
|      2013-10-01|  2025-05-05|    69309|           66239|
+----------------+------------+---------+----------------+



+-------+------------------+-------------------+
|summary|    dias_no_abrigo|idade_entrada_meses|
+-------+------------------+-------------------+
|  count|             68899|              69308|
|   mean|23.205242456349147|  15.41021815663415|
| stddev|42.860760276673275|  30.30978653200866|
|    min|                 0|                0.0|
|    25%|                 2|                0.9|
|    50%|                 7|                2.0|
|    75%|                31|               12.0|
|    max|              1745|              264.0|
+-------+------------------+-------------------+



## 16. Salvando o dataset do RA1

O resultado (~69 mil linhas) cabe tranquilo na memória, então convertemos para pandas e salvamos **um único CSV**. O `write.csv` do Spark criaria uma pasta com vários arquivos `part-*` e, no Windows, ainda exigiria o `winutils.exe` do Hadoop.

In [27]:
final = gatos.toPandas()
# inteiro com vazio vira float no pandas ("5.0"); Int64 mantém "5" e deixa o vazio
final["dias_no_abrigo"] = final["dias_no_abrigo"].astype("Int64")
final.to_csv("../data/processed/gatos_ra1.csv", index=False)

# relê o arquivo pra garantir que salvou certinho
conferencia = spark.read.csv("../data/processed/gatos_ra1.csv", header=True)
print(conferencia.count(), "linhas x", len(conferencia.columns), "colunas")
conferencia.show(3)

69309 linhas x 19 colunas
+---------+--------+--------+------------------+------------+----------+--------------+-------------------+-------------------+------------+------------+--------+------------------+-------------+--------+------------+----------------+-------------+----------------+
|animal_id|passagem|tem_nome|nome_com_asterisco|data_entrada|data_saida|dias_no_abrigo|estadia_acima_1_ano|idade_entrada_meses|faixa_etaria|        sexo|castrado|              raca|cor_principal|is_black|tipo_entrada|condicao_entrada|     desfecho|subtipo_desfecho|
+---------+--------+--------+------------------+------------+----------+--------------+-------------------+-------------------+------------+------------+--------+------------------+-------------+--------+------------+----------------+-------------+----------------+
|  A664235|       1|   False|             False|  2013-10-01|2013-10-01|             0|              False|                0.2|     filhote|desconhecido|    NULL|Domestic Short

In [28]:
spark.stop()

O arquivo `data/processed/gatos_ra1.csv` é o ponto de partida do **RA2 – Transformação e Processamento dos Dados**.